In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.jd_parser import JDParser

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data" / "candidates.jsonl"

from src.data_loader import CandidateDataLoader
from src.candidate_processor import CandidateProcessor
from src.semantic_matcher import SemanticMatcher
from src.jd_parser import JDParser
from src.ranking_engine import RankingEngine

C:\Users\anand\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
parser = JDParser()

In [4]:
jd = """
We are looking for a Backend Engineer
with at least 3 years of experience.

Required Skills:
Python
SQL
AWS
Cloud

Preferred:
Docker
Kubernetes

Excellent communication and leadership skills.
"""

In [5]:
parsed_jd = parser.parse(jd)

parsed_jd

{'raw_text': 'We are looking for a Backend Engineer\nwith at least 3 years of experience.\n\nRequired Skills:\nPython\nSQL\nAWS\nCloud\n\nPreferred:\nDocker\nKubernetes\n\nExcellent communication and leadership skills.',
 'job_title': 'Backend Engineer',
 'minimum_experience': 3,
 'required_skills': ['Python', 'SQL', 'AWS', 'Docker', 'Kubernetes', 'Cloud'],
 'preferred_skills': [],
 'soft_skills': ['Leadership', 'Communication'],
 'keywords': ['Cloud',
  'Kubernetes',
  'Docker',
  'SQL',
  'Communication',
  'Python',
  'Leadership',
  'AWS']}

In [6]:
from src.ranking_engine import RankingEngine

In [7]:
ranker = RankingEngine()

In [8]:
from src.semantic_matcher import SemanticMatcher
from src.candidate_processor import CandidateProcessor
from src.data_loader import CandidateDataLoader

In [9]:
matcher = SemanticMatcher()
processor = CandidateProcessor()
loader = CandidateDataLoader(DATA_PATH)

Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9363.30it/s]


Model loaded.


In [10]:
candidates = loader.load_candidates()

candidate = candidates[0]

profile = processor.process_profile(candidate)
skills = processor.process_skills(candidate)

candidate_text = " ".join([
    profile["headline"],
    profile["summary"],
    skills["skills_text"]
])

semantic_score = matcher.similarity(
    jd,
    candidate_text
)

ranking_result = ranker.rank(
    semantic_score,
    profile,
    skills,
    parsed_jd
)

print(ranking_result)

{'semantic_score': 0.5945, 'experience_match': True, 'experience_bonus': 0.05, 'skill_match': 0.1667, 'matched_skills': ['aws'], 'missing_skills': ['cloud', 'docker', 'kubernetes', 'python', 'sql'], 'final_score': 0.6612}


In [17]:
scores = []

for candidate in candidates[:100]:

    profile = processor.process_profile(candidate)
    skills = processor.process_skills(candidate)

    candidate_text = " ".join([
        profile["headline"],
        profile["summary"],
        skills["skills_text"]
    ])

    semantic = matcher.similarity(
        jd,
        candidate_text
    )
    
    ranking = ranker.rank(
    semantic,
    profile,
    skills,
    parsed_jd
    )
    
    scores.append({
        "candidate_id": candidate["candidate_id"],
        "name": profile["name"],
        **ranking
    })

In [18]:
ranked = sorted(
    scores,
    key=lambda x: x["final_score"],
    reverse=True
)

In [19]:
for c in ranked[:10]:
    print(
        f"{c['candidate_id']} | "
        f"{c['name']} | "
        f"Semantic={c['semantic_score']} | "
        f"Skill={c['skill_match']} | "
        f"Final={c['final_score']}"
    )

CAND_0000054 | Ved Malhotra | Semantic=0.661 | Skill=0.3333 | Final=0.7444
CAND_0000072 | Diya Sharma | Semantic=0.6742 | Skill=0.1667 | Final=0.7408
CAND_0000038 | Myra Trivedi | Semantic=0.6708 | Skill=0.1667 | Final=0.7375
CAND_0000027 | Avni Pandey | Semantic=0.6437 | Skill=0.3333 | Final=0.727
CAND_0000023 | Kavya Nair | Semantic=0.6732 | Skill=0.0 | Final=0.7232
CAND_0000067 | Shreya Bose | Semantic=0.6677 | Skill=0.0 | Final=0.7177
CAND_0000044 | Vihaan Naidu | Semantic=0.6491 | Skill=0.1667 | Final=0.7158
CAND_0000051 | Meera Arora | Semantic=0.6597 | Skill=0.0 | Final=0.7097
CAND_0000091 | Aisha Desai | Semantic=0.647 | Skill=0.0 | Final=0.697
CAND_0000043 | Aarav Sen | Semantic=0.6445 | Skill=0.0 | Final=0.6945
